In [ ]:
def compute_effdl_score(p_s, p_u, q_w, q_a, w, f,
                        ref_params=5.6e6,
                        ref_ops=2.8e8):
    """
    Compute the lab score:

    score = [1 - (p_s + p_u)] * ((q_w / 32) * w) / ref_params
          + (1 - p_s) * (max(q_w, q_a) / 32) * f / ref_ops

    Args:
        p_s (float): structured pruning ratio in [0, 1]
        p_u (float): unstructured pruning ratio in [0, 1]
        q_w (float): weight bit-width (e.g., 32, 16, 8, 1)
        q_a (float): activation bit-width (e.g., 32, 16, 8)
        w (float): number of weights (parameters)
        f (float): number of MACs
    """
    if not (0.0 <= p_s <= 1.0 and 0.0 <= p_u <= 1.0):
        raise ValueError("p_s and p_u must be in [0, 1].")
    if p_s + p_u > 1.0:
        raise ValueError("p_s + p_u must be <= 1.")
    if min(q_w, q_a, w, f) < 0:
        raise ValueError("q_w, q_a, w, and f must be non-negative.")

    params_term = (1.0 - (p_s + p_u)) * ((q_w / 32.0) * w) / ref_params
    ops_term = (1.0 - p_s) * (max(q_w, q_a) / 32.0) * f / ref_ops
    return params_term + ops_term


# Example (ResNet18 FP16 baseline should be close to 2.0):
baseline_score = compute_effdl_score(p_s=0.0, p_u=0.77, q_w=8, q_a=8, w=5.6e6, f=2.8e8)
print(f"Baseline score: {baseline_score:.4f}")